# 10 — Final AML candidate rank comparison

## Goal

Compare baseline and advanced AML screening outputs using pair-level rank changes, shared top drugs, top-pair overlap, and target-specific agreement. This is a computational agreement analysis, not biological or clinical validation.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import subprocess
import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(
            f"Required v2 API {module_name}.{attribute} is unavailable. "
            "Update v2-refactor and rerun the tests before enabling this cell."
        )
    return getattr(module, attribute)


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if BRANCH != "v2-refactor":
    raise RuntimeError(f"Expected v2-refactor, found {BRANCH!r}. v1-final is read-only.")

print(f"Project: {PROJECT_ROOT}")
print(f"Branch:  {BRANCH}")


In [ ]:
from config import SCREENING_RESULTS_DIR

BASELINE_PREDICTIONS = SCREENING_RESULTS_DIR / "baseline_predictions_full.csv"
ADVANCED_PREDICTIONS = SCREENING_RESULTS_DIR / "advanced_predictions_full.csv"
RUN_COMPARISON = False


## Steps


In [ ]:
if RUN_COMPARISON:
    import pandas as pd
    from src.screening import compare_screening_models

    baseline = pd.read_csv(BASELINE_PREDICTIONS)
    advanced = pd.read_csv(ADVANCED_PREDICTIONS)
    comparison = compare_screening_models(baseline, advanced)

    comparison.pair_comparison.to_csv(
        SCREENING_RESULTS_DIR / "baseline_advanced_pair_comparison.csv", index=False
    )
    comparison.shared_top_drugs.to_csv(
        SCREENING_RESULTS_DIR / "shared_top_drugs.csv", index=False
    )
    comparison.top_pair_overlap.to_csv(
        SCREENING_RESULTS_DIR / "top_pair_overlap.csv", index=False
    )
    comparison.target_agreement.to_csv(
        SCREENING_RESULTS_DIR / "target_specific_agreement.csv", index=False
    )
    (SCREENING_RESULTS_DIR / "comparison_summary.json").write_text(
        json.dumps(comparison.summary, indent=2, default=str), encoding="utf-8"
    )
    print(json.dumps(comparison.summary, indent=2, default=str))
else:
    print("Candidate comparison skipped: RUN_COMPARISON is False.")


## Checks

Compare identical drug–target pairs and state whether a higher score means stronger predicted binding for the relevant model target. Keep baseline and advanced raw scores separate when their training dataset/scale differs.

## Next steps

Export only compact top lists and comparison summaries for dissertation reporting. Describe all entries as computationally prioritised drug-repurposing candidates.
